**TASK 1**

In [6]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df.head())

Loaded: 3000 titles
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [7]:
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

top_ratings = ['TV-MA', 'TV-14', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(top_ratings)]


In [8]:
pivot = (
    df_filtered
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
    .pivot(index='rating', columns='decade', values='count')
    .fillna(0)
    .reindex(['TV-MA', 'R', 'TV-14', 'PG-13', 'PG'])
)


In [9]:
fig1 = px.imshow(
    pivot,
    color_continuous_scale='Blues',
    text_auto=True,
    aspect='auto',
    labels=dict(x='Release Decade', y='Content Rating', color='Titles'),
    title='Mature content (TV-MA, R) surged in the 2010s — while PG dominated before the 1990s',
)


In [10]:
max_val = pivot.to_numpy().max()
max_row, max_col = [(r, c) for r in pivot.index for c in pivot.columns if pivot.loc[r, c] == max_val][0]

fig1.add_annotation(
    x=list(pivot.columns).index(max_col),
    y=list(pivot.index).index(max_row),
    text=f"Peak: {int(max_val)}",
    showarrow=True, arrowhead=2, arrowcolor='#E63946',
    font=dict(color='#E63946', size=11), ax=55, ay=-35,
)

In [11]:
fig1.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=40, t=70, b=60),
    height=400,
)
fig1.show()

**TASK 2**

In [12]:
movies = df[(df['type'] == 'Movie') & (df['added_year'].between(2015, 2022))]

In [13]:
yearly = (
    movies.groupby('added_year')
    .size()
    .reset_index(name='count')
    .sort_values('added_year')
)

In [14]:
years      = yearly['added_year'].astype(str).tolist()
counts     = yearly['count'].tolist()
cumulative = yearly['count'].cumsum().tolist()

In [16]:
x_labels = years + ['Total']
measure  = ['relative'] * len(years) + ['total']
y_values = counts + [cumulative[-1]]
bar_text = [str(v) for v in counts] + [str(cumulative[-1])]

In [17]:
peak_idx   = counts.index(max(counts))
peak_year  = years[peak_idx]
peak_count = counts[peak_idx]

In [18]:
fig2 = go.Figure(go.Waterfall(
    orientation='v',
    x=x_labels,
    y=y_values,
    measure=measure,
    text=bar_text,
    textposition='outside',
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),   # green for additions
    totals=dict(marker_color='#2E75B6'),        # blue for total
    textfont=dict(size=11, color='#333333', family='Arial'),
))

In [19]:
fig2.add_annotation(
    x=peak_year, y=peak_count,
    text=f"<b>Biggest intake<br>{peak_count} titles</b>",
    showarrow=True, arrowhead=2, arrowcolor='#E63946',
    font=dict(color='#E63946', size=11), ax=55, ay=-45,
    bgcolor='white', bordercolor='#E63946', borderwidth=1,
)


In [20]:
fig2.update_layout(
    title="Netflix's movie library doubled in three years — but growth stalled after the peak",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=60, r=40, t=75, b=60),
    height=500,
)

In [21]:
fig2.update_xaxes(title='Year added to Netflix', showgrid=False)
fig2.update_yaxes(title='Movies added', gridcolor='#EEEEEE')
fig2.show()